# Model Evaluation Metrics

**Chapter 4: Predicting Soccer Outcomes with Classification**

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand and interpret confusion matrices
- Calculate and interpret precision, recall, and F1-score
- Use ROC curves and AUC to evaluate model performance
- Choose appropriate metrics for different soccer analytics problems
- Understand the trade-offs between different metrics

## Prerequisites

- Completed Notebooks 01-02
- Trained logistic regression model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, 
    recall_score, f1_score, roc_curve, roc_auc_score,
    classification_report
)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Load Data and Train Model

First, let's recreate our xG model from the previous notebook.

In [ ]:
# Load data
matches = sb.matches(competition_id=16, season_id=4)
events = sb.events(match_id=22912)
shots = events[events['type'] == 'Shot'].copy()

# Feature engineering
shots['goal'] = np.where(shots['shot_outcome_name'] == 'Goal', 1, 0)
shots['x'] = shots['location'].apply(lambda loc: loc[0])
shots['y'] = shots['location'].apply(lambda loc: loc[1])
shots['distance_to_goal'] = np.sqrt((120 - shots['x'])**2 + (40 - shots['y'])**2)
shots['angle_to_goal'] = np.arctan(7.32 * (120 - shots['x']) / 
                                     ((120 - shots['x'])**2 + (40 - shots['y'])**2 - (7.32/2)**2))
shots['angle_to_goal'] = np.degrees(shots['angle_to_goal'])

# Prepare data
X = shots[['distance_to_goal', 'angle_to_goal']]
y = shots['goal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

# Get predictions
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

print("Model trained and predictions made!")

## 1. Confusion Matrix

The **confusion matrix** shows the breakdown of correct and incorrect predictions.

**Four Components:**
- **True Positives (TP)**: Predicted Goal, was Goal ✓
- **True Negatives (TN)**: Predicted No Goal, was No Goal ✓
- **False Positives (FP)**: Predicted Goal, was No Goal ✗ (Type I Error)
- **False Negatives (FN)**: Predicted No Goal, was Goal ✗ (Type II Error)

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Goal', 'Goal'],
            yticklabels=['No Goal', 'Goal'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

print("\nConfusion Matrix:")
print(f"True Negatives (TN): {cm[0, 0]}")
print(f"False Positives (FP): {cm[0, 1]}")
print(f"False Negatives (FN): {cm[1, 0]}")
print(f"True Positives (TP): {cm[1, 1]}")

## 2. Accuracy

**Accuracy** = (TP + TN) / Total

Percentage of correct predictions overall.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

# Manual calculation
manual_accuracy = (cm[0, 0] + cm[1, 1]) / cm.sum()
print(f"Manual calculation: {manual_accuracy:.3f}")

**⚠️ Warning**: Accuracy can be misleading with imbalanced classes!

If 90% of shots are not goals, a model that always predicts "No Goal" gets 90% accuracy but is useless.

## 3. Precision

**Precision** = TP / (TP + FP)

Of all predictions of "Goal", what percentage were actually goals?

**Use case**: When false positives are costly (e.g., highlighting shots for video analysis)

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
print(f"Precision: {precision:.3f}")

if cm[1, 1] + cm[0, 1] > 0:
    manual_precision = cm[1, 1] / (cm[1, 1] + cm[0, 1])
    print(f"Manual calculation: {manual_precision:.3f}")
else:
    print("No positive predictions made")

print(f"\nInterpretation: {precision*100:.1f}% of predicted goals were actually goals")

## 4. Recall (Sensitivity)

**Recall** = TP / (TP + FN)

Of all actual goals, what percentage did we correctly identify?

**Use case**: When false negatives are costly (e.g., identifying all dangerous situations)

In [ ]:
recall = recall_score(y_test, y_pred, zero_division=0)
print(f"Recall: {recall:.3f}")

if cm[1, 1] + cm[1, 0] > 0:
    manual_recall = cm[1, 1] / (cm[1, 1] + cm[1, 0])
    print(f"Manual calculation: {manual_recall:.3f}")

print(f"\nInterpretation: We caught {recall*100:.1f}% of all actual goals")

## 5. F1-Score

**F1-Score** = 2 × (Precision × Recall) / (Precision + Recall)

The **harmonic mean** of precision and recall. Balances both metrics.

**Use case**: When you want a single metric that considers both precision and recall

In [ ]:
f1 = f1_score(y_test, y_pred, zero_division=0)
print(f"F1-Score: {f1:.3f}")

if precision + recall > 0:
    manual_f1 = 2 * (precision * recall) / (precision + recall)
    print(f"Manual calculation: {manual_f1:.3f}")

## 6. Classification Report

Scikit-learn provides a convenient summary of all metrics:

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Goal', 'Goal'], zero_division=0))

## 7. ROC Curve and AUC

**ROC (Receiver Operating Characteristic) Curve**:
- Shows trade-off between True Positive Rate (Recall) and False Positive Rate
- Plots performance at different probability thresholds

**AUC (Area Under Curve)**:
- Single number summarizing ROC curve
- Range: 0 to 1 (higher is better)
- 0.5 = random guessing, 1.0 = perfect classifier

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve for xG Model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nAUC Score: {auc:.3f}")

## Summary

In this notebook, we learned:

1. ✅ **Confusion Matrix**: Shows all types of correct and incorrect predictions
2. ✅ **Accuracy**: Overall correctness (can be misleading with imbalanced data)
3. ✅ **Precision**: How many predicted positives were actually positive
4. ✅ **Recall**: How many actual positives we correctly identified
5. ✅ **F1-Score**: Harmonic mean of precision and recall
6. ✅ **ROC/AUC**: Overall model performance across all thresholds

## Key Takeaways

- **Don't rely on accuracy alone** for imbalanced data
- **Precision vs Recall** trade-off depends on your use case
- **AUC** is threshold-independent and good for comparing models
- **Choose metrics** based on the cost of different types of errors

## Next Steps

In the next notebook, we'll explore **K-Nearest Neighbors (KNN)**, another classification algorithm!

## Practice Exercises

1. **Calculate Specificity**: TN / (TN + FP) - the recall for the negative class
2. **Different Threshold**: Try threshold=0.3 instead of 0.5 and recalculate metrics
3. **Imbalanced Example**: Create a dataset with 95% non-goals and see how metrics change
4. **Compare Models**: Train a second model with different features and compare AUC
5. **Cost Analysis**: If FN costs $100 and FP costs $10, what's the total cost of errors?